# Lab | Chains in LangChain (LangChain 1.x)

Updated for current LangChain. The legacy classes (`LLMChain`, `SimpleSequentialChain`,
`SequentialChain`, `MultiPromptChain`) were removed from `langchain` 1.x, so each section
uses the modern equivalent built with the `|` (LCEL) pipe syntax.

## Outline

* Basic chain (replaces LLMChain)
* Sequential Chains
  * Simple sequential chain (replaces SimpleSequentialChain)
  * Multi-output sequential chain (replaces SequentialChain)
* Router Chain (replaces MultiPromptChain)

In [1]:
# Run once, then restart the kernel (Kernel -> Restart)
%pip install -U langchain langchain-core langchain-openai python-dotenv pandas

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.7 MB 5.6 MB/s eta 0:00:02
   ----- ---------------------------------- 1.6/10.7 MB 4.1 MB/s eta 0:00:03
   ------- -------------------------------- 2.1/10.7 MB 3.9 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/10.7 MB 3.4 MB/s eta 0:00:03
   ---------- ----------------------------- 2.9/10.7 MB 3.1 MB/s eta 0:00:03
   ------------ --------------------------- 3.4/10.7 MB 2.8 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/10.7 MB 2.6 MB/s eta 0:00:03
   --------------- ------------------------ 4.2/10.7 MB 2.5 MB/s eta 0:00:03
   ---------------- ----------------------- 4.5/10.7 MB 2.4 MB/s eta 0:00:03
   ------------------ --------------------- 5.0/10.7 MB 2.2 MB/s eta 0:00:03
   ------------------- -------------------- 5.2/10.7 MB 2.2 MB/s eta 0:00:03
   -------------------- ------------------- 5.5/10.7 MB 2.1 MB/s eta 0:00:03
   ---

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.14.1 which is incompatible.


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [4]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [5]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## Basic chain (replaces LLMChain)

`LLMChain(llm=llm, prompt=prompt)` is now `prompt | llm | StrOutputParser()`,
and `chain.run(x)` is now `chain.invoke({...})`.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

MODEL = "gpt-6.1-sol"  # change to any OpenAI chat model you have access to

In [8]:
#Replace None by your own value and justify
llm = ChatOpenAI(model=MODEL, temperature=None)

In [9]:
prompt = ChatPromptTemplate.from_template(
    # TODO: Write a query that would take a variable to describe any product
    "TODO: your prompt here, using the variable {product}"
)

In [10]:
chain = prompt | llm | StrOutputParser()

In [11]:
product = None  # TODO: Select a product type to be described
chain.invoke({"product": product})

'What should the prompt accomplish, and how should it use `None`—as a literal value, a placeholder, or a variable name?'

## Simple sequential chain (replaces SimpleSequentialChain)

The output of the first chain is passed as the input of the second chain.

In [16]:
llm = ChatOpenAI(model=MODEL, temperature=1)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    # TODO: Repeat the initial query or create a new query that would feed into the second prompt
    "TODO: your first prompt here, using the variable {product}"
)

# Chain 1
chain_one = first_prompt | llm | StrOutputParser()

In [17]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    # TODO: Write the second prompt query that takes an input variable whose input will come from the previous prompt
    "TODO: your second prompt here, using the variable {first_output}"
)
# chain 2
chain_two = second_prompt | llm | StrOutputParser()

In [18]:
def show_and_pass(text):
    # mimics verbose=True: prints the intermediate result
    print(f"[chain one output]\n{text}\n")
    return {"first_output": text}

overall_simple_chain = chain_one | RunnableLambda(show_and_pass) | chain_two

In [19]:
overall_simple_chain.invoke({"product": product})

[chain one output]
What should the prompt accomplish, and how would you like `None` to be used?



'What would you like the second prompt to accomplish? Also, should `None` be treated as a literal value, a placeholder to replace, or an indication that no value was provided?'

**Repeat the above twice for different products**

## Multi-output sequential chain (replaces SequentialChain)

`RunnablePassthrough.assign(name=chain)` runs a chain and adds its result to the
dictionary under `name` (the old `output_key`), so later chains can use every earlier output.

In [32]:
llm = ChatOpenAI(model=MODEL, temperature=1)

first_prompt = ChatPromptTemplate.from_template(
    # TODO: This prompt should translate a review
    "TODO: your translation prompt here, using the variable {Review}"
)

chain_one = first_prompt | llm | StrOutputParser()
# output name: English_Review (change it if you like, and update the prompts below)

In [33]:
second_prompt = ChatPromptTemplate.from_template(
    # TODO: Write a prompt to summarize a review
    "TODO: your summary prompt here, using the variable {English_Review}"
)

chain_two = second_prompt | llm | StrOutputParser()
# output name: summary

In [28]:
# prompt template 3: detect the language of the original review
third_prompt = ChatPromptTemplate.from_template(
    "TODO: your language-detection prompt here, using the variable {Review}"
)
# chain 3: input= Review and output= language
chain_three = third_prompt | llm | StrOutputParser()

In [29]:
# prompt template 4: follow up message that takes as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "TODO: your follow-up prompt here, using the variables {summary} and {language}"
)
chain_four = fourth_prompt | llm | StrOutputParser()
# output name: followup_message

In [30]:
# overall_chain: input= Review
# and output= English_Review, summary, language, followup_message
overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_one)
    | RunnablePassthrough.assign(summary=chain_two)
    | RunnablePassthrough.assign(language=chain_three)
    | RunnablePassthrough.assign(followup_message=chain_four)
)

In [31]:
review = df.Review[5]
result = overall_chain.invoke({"Review": review})
result

{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'English_Review': 'I find the taste mediocre. The foam doesn’t last, which is strange. I buy the same ones in stores, and they taste much better…\n\nOld stock or counterfeit!?',
 'summary': 'The product tastes worse than the same one bought in stores, and the foam disappears quickly, raising concerns that it may be old stock or counterfeit.',
 'language': 'French (fr)',
 'followup_message': 'Je comprends vos inquiétudes concernant le goût inhabituel et la mousse qui disparaît rapidement. Pour mieux comprendre le problème, pourriez-vous préciser où et quand vous avez acheté ce produit, ainsi que son numéro de lot et la date indiquée sur l’emballage ? Avez-vous également remarqué des différences d’emballage par rapport au même produit acheté en magasin ?'}

**Repeat the above twice for different products or reviews**

## Router Chain (replaces MultiPromptChain)

The router asks the model to pick a destination using structured output
(no JSON parsing needed), then sends the question to the matching chain.

In [34]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [35]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [36]:
from typing import Literal
from pydantic import BaseModel, Field

In [43]:
llm = ChatOpenAI(model=MODEL, temperature=1)

In [44]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = prompt | llm | StrOutputParser()
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [45]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

In [46]:
# The router returns a structured object instead of a JSON code block
DestinationName = Literal[tuple([p["name"] for p in prompt_infos] + ["DEFAULT"])]

class RouteQuery(BaseModel):
    destination: DestinationName = Field(
        description='Name of the prompt to use, or "DEFAULT" if none is a good fit'
    )
    next_inputs: str = Field(
        description="A potentially modified version of the original input"
    )

ROUTER_TEMPLATE = """Given a raw text input to a language model, select the model \
prompt best suited for the input. You will be given the names of the available \
prompts and a description of what each prompt is best suited for. You may also \
revise the original input if you think that revising it will ultimately lead to \
a better response from the language model. Use "DEFAULT" if the input is not well \
suited for any of the candidate prompts.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{input}"""

router_prompt = ChatPromptTemplate.from_template(ROUTER_TEMPLATE).partial(
    destinations=destinations_str
)

router_chain = router_prompt | llm.with_structured_output(RouteQuery)

In [47]:
def route(x):
    decision = x["route"]
    print(f"[router] -> {decision.destination}: {decision.next_inputs}")  # mimics verbose=True
    target = destination_chains.get(decision.destination, default_chain)
    return target.invoke({"input": decision.next_inputs})

chain = RunnablePassthrough.assign(route=router_chain) | RunnableLambda(route)

In [48]:
chain.invoke({"input": "What is black body radiation?"})

[router] -> physics: What is black body radiation?


'**Blackbody radiation** is the electromagnetic radiation emitted by an ideal object—a *blackbody*—that absorbs all light and other radiation falling on it.\n\nIts emitted spectrum depends **only on its temperature**, not on its material. As the temperature rises:\n- It emits more energy overall.\n- The peak of its spectrum shifts toward shorter wavelengths.\n\nFor example, a warm object emits mainly infrared radiation, which is invisible to our eyes. A sufficiently hot object glows red, then orange, and eventually white.\n\nReal objects are not perfect blackbodies, but many approximate them. Stars, for example, have roughly blackbody-like spectra.\n\nExplaining this spectrum led Max Planck to propose that energy is emitted and absorbed in discrete packets called **quanta**, helping launch quantum physics.'

In [49]:
chain.invoke({"input": "what is 2 + 2"})

[router] -> math: What is 2 + 2?


'2 + 2 = 4.'

In [50]:
chain.invoke({"input": "Why does every cell in our body contain DNA?"})

[router] -> biology: Why do most cells in our body contain DNA, what does it do, and are there exceptions?


'Most cells in our body contain **DNA because it stores the genetic instructions cells need to function and maintain themselves**. When a cell divides, it usually copies its DNA so that each daughter cell receives a set of those instructions.\n\n### What does DNA do?\nDNA contains genes—sequences that cells use to make RNA. Many RNAs guide the production of **proteins**, which build cell structures, carry signals, transport substances, and carry out chemical reactions. Other RNAs perform important jobs directly.\n\nDNA also contains regulatory sequences that help control when and where genes are used. Nearly all your cells have essentially the same DNA, but they activate different sets of genes. That is why a nerve cell and a muscle cell can have very different shapes and functions.\n\nMost human DNA is stored in the **nucleus**. A small additional amount is found in **mitochondria**, the structures that produce much of a cell’s usable energy.\n\n### Are there exceptions?\nYes. Some sp

**Repeat the above at least once for different inputs and chains executions - Be creative!**

### Exercise solution

Five new questions, one aimed at each kind of route. The router should send them to
History, computer science, biology, math, and finally DEFAULT for a question
that fits none of the experts.

In [51]:
# History route
chain.invoke({"input": "Why did the Berlin Wall fall in 1989, and what happened in the months right after?"})

[router] -> History: Why did the Berlin Wall fall in 1989, and what happened in the months right after?


'The Berlin Wall fell in November 1989 because **East Germany’s communist government lost the ability to keep its people confined—and could no longer count on Soviet force to preserve its rule**. Years of economic and political frustration produced a crisis of mass emigration and peaceful protest. A confused announcement on 9 November then turned that crisis into the opening of the border.\n\n## Why did the Wall fall?\n\n### 1. The Soviet Union would no longer guarantee communist rule by force\n\nThe Wall, built in 1961, had stopped East Germans fleeing to West Berlin. Its survival depended ultimately on Soviet support for East Germany’s ruling Socialist Unity Party, or SED.\n\nAfter taking power in 1985, **Mikhail Gorbachev** introduced reforms in the Soviet Union and increasingly allowed Eastern European governments to determine their own futures. Unlike earlier Soviet leaders, he was unwilling to use military intervention to keep them in power.\n\nThis mattered enormously. Soviet fo

In [52]:
# Computer science route
chain.invoke({"input": "When would you pick a hash map over a balanced binary search tree? Compare time and space complexity."})

[router] -> computer science: When would you pick a hash map over a balanced binary search tree? Compare their average-case and worst-case time complexities for lookup, insertion, and deletion, as well as space complexity. Discuss ordering, range queries, and practical trade-offs.


'Pick a **hash map** when you primarily need fast exact-key lookups, insertions, and deletions, and do not need keys kept in sorted order. Pick a **balanced binary search tree**—such as an AVL or red-black tree—when you need sorted traversal, range queries, or predictable worst-case performance.\n\n### Time complexity\n\nFor \\(n\\) stored entries:\n\n| Operation | Hash map: average / expected* | Hash map: worst case | Balanced BST: average | Balanced BST: worst case |\n|---|---:|---:|---:|---:|\n| Lookup | \\(O(1)\\) | \\(O(n)\\) | \\(O(\\log n)\\) | \\(O(\\log n)\\) |\n| Insertion | \\(O(1)\\) amortized | \\(O(n)\\) | \\(O(\\log n)\\) | \\(O(\\log n)\\) |\n| Deletion | \\(O(1)\\) | \\(O(n)\\) | \\(O(\\log n)\\) | \\(O(\\log n)\\) |\n\n\\*Hash-map bounds assume a well-distributed hash function and a controlled load factor. Resizing makes insertion constant-time **amortized**, not constant-time for every individual operation. Deletion may also trigger resizing in some implementations, 

In [ ]:
# Biology route
chain.invoke({"input": "How do tardigrades survive being completely dried out for years?"})

In [ ]:
# Math route (with a word problem the router may rephrase)
chain.invoke({"input": "A train leaves at 9:00 going 80 km/h, a second leaves the same station at 10:00 going 120 km/h. When does the second catch up?"})

In [ ]:
# DEFAULT route: none of the experts fit
chain.invoke({"input": "Suggest a name for a cozy coffee shop that also sells used books."})

### Extra: add a new expert and run a batch

A new destination only needs a template and an entry in `prompt_infos`; then rebuild the
router so it knows about it. `chain.batch(...)` runs several questions in parallel.

In [53]:
cooking_template = """You are an experienced chef and food scientist. \
You explain cooking techniques clearly and tell people why they work, \
not just what to do.

Here is a question:
{input}"""

prompt_infos.append({
    "name": "cooking",
    "description": "Good for answering cooking and food questions",
    "prompt_template": cooking_template,
})

# rebuild the destinations and the router with the new expert
destination_chains["cooking"] = ChatPromptTemplate.from_template(cooking_template) | llm | StrOutputParser()
destinations_str = "\n".join(f"{p['name']}: {p['description']}" for p in prompt_infos)
DestinationName = Literal[tuple([p["name"] for p in prompt_infos] + ["DEFAULT"])]

class RouteQuery(BaseModel):
    destination: DestinationName = Field(
        description='Name of the prompt to use, or "DEFAULT" if none is a good fit'
    )
    next_inputs: str = Field(
        description="A potentially modified version of the original input"
    )

router_prompt = ChatPromptTemplate.from_template(ROUTER_TEMPLATE).partial(destinations=destinations_str)
router_chain = router_prompt | llm.with_structured_output(RouteQuery)
chain = RunnablePassthrough.assign(route=router_chain) | RunnableLambda(route)

In [54]:
questions = [
    "Why does searing meat make it taste better?",
    "What is the derivative of x^3 * sin(x)?",
    "Who was Ada Lovelace and why is she important?",
]
answers = chain.batch([{"input": q} for q in questions])
for q, a in zip(questions, answers):
    print(f"Q: {q}\nA: {a}\n" + "-" * 60)

[router] -> cooking: Why does searing meat make it taste better?
[router] -> math: What is the derivative of x^3 * sin(x)?
[router] -> History: Who was Ada Lovelace, and why is she important in the history of computing?
Q: Why does searing meat make it taste better?
A: Searing makes meat taste better mainly because it creates **new flavor compounds on the surface**, along with a browned crust that contrasts with the tender interior.

### The main reason: the Maillard reaction
When a relatively dry meat surface gets hot—browning becomes rapid around **140–165°C / 285–330°F**—amino acids and naturally occurring sugars react. This is the **Maillard reaction**, which produces hundreds of compounds responsible for roasted, nutty, savory, and “meaty” aromas.

Those aromas matter enormously: much of what we experience as flavor comes from smell. A gently poached steak and a seared steak can have equally tender interiors, but the seared one has an extra layer of aromatic complexity.

### Textu